In [2]:
import sys
!{sys.executable} -m pip install groq

  Using cached groq-1.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
Using cached groq-1.1.2-py3-none-any.whl (141 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)

   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   -------------------- ------------------- 1/2 [groq]
   ---------------------------------------- 2/2 [groq]




[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [5]:
from groq import Groq

client = Groq(api_key="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")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # ← updated model name
    messages=[{"role": "user", "content": "Say hi!"}]
)

print(response.choices[0].message.content)
print(" Groq Connected Successfully!")

Hi. How can I help you today!
 Groq Connected Successfully!


In [6]:
import pandas as pd

df = pd.read_csv('../data/customer_features.csv')

print(" Data loaded!")
print("Shape:", df.shape)
df.head(3)

 Data loaded!
Shape: (594, 9)


,Customer_ID,Recency,Frequency,Monetary,Avg_Order_Value,Total_Quantity,Fav_Category,Region,Churned
0,C0001,130,5,4232,846.400000,10,Electronics,North,1
1,C0002,132,3,4234,1411.333333,6,Office,North,1
2,C0003,107,7,5641,805.857143,8,Accessories,East,1


In [7]:
def explain_churn_risk(customer_id):
    customer = df[df['Customer_ID'] == customer_id].iloc[0]
    
    prompt = f"""
    You are a business analyst AI. Analyze this customer data 
    and explain their churn risk in simple business language.
    
    Customer Data:
    - Customer ID: {customer['Customer_ID']}
    - Days Since Last Purchase: {customer['Recency']} days
    - Total Orders: {customer['Frequency']}
    - Total Revenue: ${customer['Monetary']}
    - Average Order Value: ${round(customer['Avg_Order_Value'], 2)}
    - Favourite Category: {customer['Fav_Category']}
    - Region: {customer['Region']}
    - Churn Status: {'High Risk' if customer['Churned'] == 1 else 'Low Risk'}
    
    Provide:
    1. Churn Risk Assessment (High/Medium/Low)
    2. Key reasons for this assessment
    3. Top 2 business recommendations
    
    Keep response clear and concise in 5-6 lines.
    """
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

print(" Churn explainer ready!")

 Churn explainer ready!


In [8]:
test_customers = ['C0001', 'C0050', 'C0100']

for customer_id in test_customers:
    print(f"\n{'='*55}")
    print(f"  Customer: {customer_id}")
    print(f"{'='*55}")
    result = explain_churn_risk(customer_id)
    print(result)


  Customer: C0001
Churn Risk Assessment: High 
Key reasons: 130 days since last purchase and limited total orders (5). 
Recommendations: 
1. Targeted marketing to Electronics category in North region.
2. Offer personalized promotions to re-engage customer C0001.

  Customer: C0050
Churn Risk Assessment: Low 
Key reasons: Recent activity and moderate purchase history.
Recommendations: 
1. Personalized Electronics offers to East region customers.
2. Exclusive loyalty programs to retain high-value customers.

  Customer: C0100
Churn Risk Assessment: High 
Key reasons: Inactive for 256 days and only 2 total orders.
Recommendations: 
1. Targeted marketing to favourites.
2. Personalized offers to reactivate purchase.


In [9]:
def ask_data_question(question):
    summary = f"""
    Dataset Summary:
    - Total Customers: {df.shape[0]}
    - Churned Customers: {df['Churned'].sum()}
    - Churn Rate: {round(df['Churned'].mean() * 100, 2)}%
    - Avg Days Since Purchase: {round(df['Recency'].mean(), 2)}
    - Avg Orders Per Customer: {round(df['Frequency'].mean(), 2)}
    - Avg Revenue Per Customer: ${round(df['Monetary'].mean(), 2)}
    - Top Category: {df['Fav_Category'].mode()[0]}
    - Top Region: {df['Region'].mode()[0]}
    - High Risk Customers: {df[df['Churned']==1].shape[0]}
    - Low Risk Customers: {df[df['Churned']==0].shape[0]}
    """
    
    prompt = f"""
    You are a data analyst AI. Answer this business question 
    based on the customer churn dataset below.
    
    {summary}
    
    Question: {question}
    
    Give a clear actionable business answer in 3-4 lines.
    """
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

print(" Q&A function ready!")

 Q&A function ready!


In [10]:
questions = [
    "What is the overall churn rate and is it concerning?",
    "Which type of customers are most at risk of churning?",
    "What strategy would you recommend to improve retention?"
]

for question in questions:
    print(f"\n{'='*55}")
    print(f"Q: {question}")
    print(f"{'='*55}")
    answer = ask_data_question(question)
    print(f"A: {answer}")


Q: What is the overall churn rate and is it concerning?
A: The overall churn rate is 23.74%, which is concerning as it indicates that nearly a quarter of the customer base has stopped doing business with the company. This rate suggests a significant loss of revenue and customer loyalty. To mitigate this, the company should prioritize retention strategies and investigate the root causes of churn. Immediate action is recommended to prevent further customer loss.

Q: Which type of customers are most at risk of churning?
A: Based on the dataset, high-risk customers are most at risk of churning, with 141 customers already identified as high risk. These customers require immediate attention to prevent churn. Targeted retention strategies should be implemented to retain these high-risk customers. This will help reduce the overall churn rate of 23.74%.

Q: What strategy would you recommend to improve retention?
A: To improve retention, I recommend targeting the high-risk customer segment with

In [11]:
def generate_business_report():
    prompt = f"""
    You are a senior business analyst. Generate a professional 
    churn analysis report based on this data:
    
    - Total Customers: {df.shape[0]}
    - Churned: {df['Churned'].sum()} ({round(df['Churned'].mean()*100,2)}%)
    - Retained: {(df['Churned']==0).sum()} ({round((1-df['Churned'].mean())*100,2)}%)
    - Avg Days Since Purchase: {round(df['Recency'].mean(),2)} days
    - Avg Orders Per Customer: {round(df['Frequency'].mean(),2)}
    - Avg Revenue Per Customer: ${round(df['Monetary'].mean(),2)}
    - Most Common Category: {df['Fav_Category'].mode()[0]}
    - Most Common Region: {df['Region'].mode()[0]}
    
    Write a professional report with these sections:
    1. Executive Summary
    2. Key Findings
    3. Risk Assessment
    4. Business Recommendations
    
    Keep it professional and concise.
    """
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

# Generate report
print("Generating AI Business Report...")
print("="*60)
report = generate_business_report()
print(report)

# Save report
with open('../data/AI_Business_Report.txt', 'w') as f:
    f.write(report)

print("\n Report saved as AI_Business_Report.txt!")

Generating AI Business Report...
**Churn Analysis Report**

**1. Executive Summary**

This churn analysis report provides an overview of customer retention and churn trends for our business. Based on the data, we have a customer base of 594, with 76.26% of customers retained and 23.74% churned. The report highlights key findings, assesses risks, and provides business recommendations to improve customer retention and revenue growth.

**2. Key Findings**

The analysis reveals the following key insights:

* The average customer has been with us for approximately 65.63 days, indicating a relatively short tenure.
* Customers place an average of 5.05 orders, generating an average revenue of $6,287.55 per customer.
* Electronics is the most popular product category, and the East region accounts for the largest customer base.
* The customer churn rate is 23.74%, which is a significant proportion of our customer base.

**3. Risk Assessment**

The high churn rate of 23.74% poses a significant ri